# EN Transformer NER Training on Colab GPU

**重要**: ランタイム → ランタイムのタイプを変更 → **T4 GPU** → 保存 してから実行

- `spacy[cuda12x]` は使わない（Colabのcupyを壊す）
- `spacy init config --gpu` で公式GPU configを生成
- `pip install spacy spacy-transformers` のみ

In [ ]:
# Step 1: Install & verify GPU
!pip install -q spacy spacy-transformers
!nvidia-smi
import spacy, thinc.util
print(f'spacy={spacy.__version__}, cupy={thinc.util.has_cupy}')
assert thinc.util.has_cupy, 'CuPy not available! Check GPU runtime.'

In [ ]:
# Step 2: Clone repo & generate GPU config
!git clone https://github.com/plenoai/pleno-anonymize.git 2>&1 | tail -3
%cd /content/pleno-anonymize/packages/training
!python -m spacy init config --lang en --pipeline transformer,ner --gpu /tmp/base.cfg
!python -m spacy init fill-config /tmp/base.cfg configs/train_transformer_en_gpu.cfg

In [ ]:
# Step 3: Fetch training data & train
!git fetch origin tmp/en-data 2>&1 | tail -1
!git checkout origin/tmp/en-data -- data/processed/en/
!ls data/processed/en/
!python -m spacy train configs/train_transformer_en_gpu.cfg \
    --output output/en-transformer \
    --paths.train data/processed/en/train.spacy \
    --paths.dev data/processed/en/dev.spacy \
    --gpu-id 0

In [ ]:
# Step 4: Show results
!cat output/en-transformer/model-best/meta.json

In [ ]:
# Step 5: Evaluate on test set
!python -m spacy evaluate output/en-transformer/model-best data/processed/en/test.spacy --gpu-id 0